# NRTK Configuration & Testbed Notebook

This notebook will discuss the process of running the Model Evaluation Configuration app & the Model Evaluation Testbed app.

## Table of Contents

1. [Model Evaluation Configuration](#model-evaluation-configuration)
2. [Model Evaluation Testbed](#model-evaluation-testbed)

## Model Evaluation Configuration

- To run the model configuration panel app you can use the model_evaluation_configuration.ipynb in the docs/how-tos folder.
- Or by using the following code.

In [ ]:
# load jatic_ri configuration app.
from jatic_ri._common._panel.configurations.model_evaluation_configuration import ModelEvaluationConfigApp

In [ ]:
# Set task to either 'image_classification' or 'object_detection'
task = 'object_detection'
# Local to true for local deployment, false for remote deployment
local = True

In [ ]:
# Run app to view
app = ModelEvaluationConfigApp(task=task, local=local)
app.panel().servable()

Select the NRTK checkbox and click next, or use the code below to select next for you.

In [ ]:
# programmatically move between stages (as long as the button isn't disabled)
app.pipeline._select_next
if not app.pipeline.next_button.disabled:
    app.pipeline.next_button.clicks += 1
else:
    print('There is no next stage.')

In [ ]:
# programmatically move between stages (as long as the button isn't disabled)
if not app.pipeline.prev_button.disabled:
    app.pipeline.prev_button.clicks += 1
else: 
    print('There is no previous stage.')

From here you can select the values desired for each perturber you want to set.
After you have selected a pertuber factory, factory type, and the desired configurations. 'Click Add Test Stage'
This will add that to the list of perturber factories to use.
Once you have selected all the configurations you would like click next and it will produce a configuration.json file.

In [ ]:
# Here is an example of the configuration.json that the app generates.
input_config ={
    "task": "object_detection",
    "NRTKApp_0": {
        "TYPE": "NRTKTestStage",
        "CONFIG": {
            "name": "natural_robustness_brightnessperturber",
            "perturber_factory": {
                "type": "nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory",
                "nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory": {
                    "perturber": "nrtk.impls.perturb_image.generic.PIL.enhance.BrightnessPerturber",
                    "theta_key": "factor",
                    "start": 0.5,
                    "stop": 2,
                    "step": 6
                }
            }
        }
    }
}

## Model Evaluation Testbed

- To run the model configuration panel app you can use the Model_Evaluation_Testbed.ipynb in the docs/how-tos folder.
- Or by using the following code.

In [ ]:
from jatic_ri._common._panel.dashboards.model_evaluation_dashboard import ModelEvaluationTestbed
import json
import os
from pathlib import Path

In [ ]:
# Set the task to either 'image_classification' or 'object_detection'
task = 'object_detection'
# Set output directory for the report
report_output_dir = 'runtime_output'

# instantiate the app
app = ModelEvaluationTestbed(
    task=task,
    output_dir=report_output_dir,
)
app.panel().servable()

In [ ]:
# Take the configuration from the Model Evaluation Configuration app and set it to the input_config variable
app.config_file.value = json.dumps(input_config)

# Set the dataset and directory
visdrone_dataset_dir = Path('/path/to/visdrone/dataset') # Replace with the actual path to your dataset
app.dataset_1_selector.value = "Visdrone dataset"
app.dataset_1_directory.value = str(visdrone_dataset_dir)

# Set Model and model weights path
visualized_model_name = {value: key for key, value in app.model_label_map.items()}["resnet18"]
app.model_widgets['Model 1 type']['model_selector'].value = visualized_model_name
app.model_widgets['Model 1 type']['model_weights_path'].value = "visdrone_weights"

# this is a rule-of-thumb for a 'good' model when using the mAP-50 metric
app.threshold = "0.50" 

# trigger computation
app.run_analysis_button.clicks += 1 